In [1]:
print("hello")

hello


In [1]:
import os

os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["hadoop.home.dir"] = r"C:\hadoop"
os.environ["PATH"] += r";C:\hadoop\bin"

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, from_json, to_timestamp
from pyspark.sql.types import *

spark = SparkSession.builder \
    .appName("OrderStreamingNotebook") \
    .master("local[*]") \
    .config(
        "spark.jars.packages",
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0"
    ) \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

print("Spark session started")

Spark session started


In [2]:
order_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("event_type", StringType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("price", DoubleType(), True),
    StructField("event_time", StringType(), True)
])

print("Schema created")

Schema created


In [3]:

#Reads live events continuously from Kafka topic order_events.

df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "127.0.0.1:9092") \
    .option("subscribe", "order_events") \
    .option("startingOffsets", "latest") \
    .option("failOnDataLoss", "false") \
    .load()

print("Kafka stream connected")

Kafka stream connected


In [4]:

#Converts Kafka key/value bytes into readable string format.
raw_df = df.selectExpr(
    "CAST(key AS STRING) as kafka_key",
    "CAST(value AS STRING) as json_data",
    "timestamp as kafka_timestamp"
)

print("Raw dataframe created")

Raw dataframe created


In [5]:

#Safely converts JSON string into structured Spark columns using schema.
parsed_df = raw_df.select(
    "kafka_key",
    "kafka_timestamp",
    from_json(col("json_data"), order_schema).alias("order_data")
)

print("JSON parsing applied")

JSON parsing applied


In [6]:

#Extracts individual JSON fields into normal dataframe columns.

orders_df = parsed_df.select(
    "kafka_key",
    "kafka_timestamp",
    col("order_data.order_id").alias("order_id"),
    col("order_data.customer_id").alias("customer_id"),
    col("order_data.product_id").alias("product_id"),
    col("order_data.event_type").alias("event_type"),
    col("order_data.quantity").alias("quantity"),
    col("order_data.price").alias("price"),
    col("order_data.event_time").alias("event_time")
)

print("Fields flattened")

Fields flattened


In [7]:
#Converts string event time into timestamp for event-time processing.

events_df = orders_df.withColumn(
    "event_timestamp",
    to_timestamp(col("event_time"))
)

print("Event-time semantics applied")

Event-time semantics applied


In [8]:
query = events_df.writeStream \
    .format("memory") \
    .queryName("events_stream") \
    .outputMode("append") \
    .start()

print("Memory stream started")

Memory stream started


In [9]:
query.awaitTermination(15)

False

In [27]:
spark.sql("SELECT * FROM events_stream LIMIT 20").show(truncate=False)

+---------+-----------------------+--------+-----------+----------+----------+--------+------+--------------------------+--------------------------+
|kafka_key|kafka_timestamp        |order_id|customer_id|product_id|event_type|quantity|price |event_time                |event_timestamp           |
+---------+-----------------------+--------+-----------+----------+----------+--------+------+--------------------------+--------------------------+
|NULL     |2026-03-31 13:47:58.905|order_31|cust_2     |prod_54   |CREATED   |1       |157.24|2026-03-31T08:17:36.905127|2026-03-31 08:17:36.905127|
|NULL     |2026-03-31 13:47:59.418|order_30|cust_16    |prod_84   |CANCELLED |2       |155.84|2026-03-31T08:16:53.418166|2026-03-31 08:16:53.418166|
|NULL     |2026-03-31 13:47:59.935|order_42|cust_12    |prod_46   |CREATED   |1       |70.5  |2026-03-31T08:17:05.935365|2026-03-31 08:17:05.935365|
|NULL     |2026-03-31 13:48:00.449|order_38|cust_3     |prod_96   |CREATED   |4       |188.0 |2026-03-31T0